## PROJECT #4 DEMO: HYBRID NAVIGATION SIMULATION
#### Notebook version
---------------------------------------------
This script serves as the "Simulation Layer" for the Hybrid Controller.
It handles the physics, visual rendering, and sensor simulation, while delegating
the actual control logic to the 'unicycle_controllers.py' library.

Usage:
    
    Ensure 'unicycle_controllers.py' is in the same directory.
    Run code cells, if running all at once ensure there are enought "time.sleep(1)" lines to avoid crashing

#### Deprecated part:
Scenario ("Safe-Suicide-Safe"):
1. The Planner wants to go straight through a wall (Waypoint 2 to 3).
2. The Robot must detect this "Suicide Path" using simulated sensors.
3. The Reactive layer must override the planner to follow the wall safely.
4. Once the obstacle is cleared, the robot must rejoin the path.


In [1]:
# imports
from __future__ import annotations
from dataclasses import dataclass, field
from typing import Sequence, Tuple, List, Dict
import numpy as np
import roboticstoolbox as rtb
import spatialgeometry as sg
import spatialmath as sm
import swift
import time
import math
import matplotlib.pyplot as plt
import matplotlib as mtb

In [24]:
# Set default matplotlib params
mtb.rcParams.update(mtb.rcParamsDefault)

# --- IMPORT THE BRAIN ---
# We import the controller logic from the external file.
# This enforces Separation of Concerns: Logic vs. Simulation.
from unicycle_controllers import HybridNavigator

# --- SIMULATION CONSTANTS & COLORS ---
FLOOR_HEIGHT = 0.02
COLOR_BLUE = (0.2, 0.5, 0.8, 1.0)   # Robot color
COLOR_WALL = (0.8, 0.3, 0.1, 1.0)   # Wall color # was gray: (0.3, 0.3, 0.3, 1.0)
COLOR_RAY = (1.0, 0.8, 0.0, 0.7)    # Sensor ray color (Yellow)
COLOR_GREEN = (0.2, 0.8, 0.3, 0.7)  # Waypoint/Target color
COLOR_GREY = (0.5, 0.5, 0.5, 0.8)   # Path line color
COLOR_LINE = (0.2, 0.6, 0.3, 1.0)    # Alternate path colour

### --- VISUALIZATION & PHYSICS WRAPPERS ---


In [3]:
@dataclass
class AgentVisual:
    """
    A wrapper class for the 3D visual object (Cuboid) representing the robot.
    It updates the position of the 3D block in the Swift browser simulator.
    """
    body: sg.Cuboid

    def update(self, base_pose: sm.SE3) -> None:
        """Moves the visual block to the robot's new physical pose."""
        # Offset by FLOOR_HEIGHT so the robot sits on top of the floor, not inside it
        self.body.T = base_pose * sm.SE3(0.0, 0.0, 0.05 + FLOOR_HEIGHT)

def make_agent(c):
    """Factory function to create the visual block (chassis)."""
    return AgentVisual(sg.Cuboid(scale=[0.4, 0.2, 0.1], color=c))

def state_to_pose(s):
    """Converts state vector [x, y, theta] into an SE3 Homogeneous Transformation Matrix."""
    return sm.SE3(s[0], s[1], 0.0) * sm.SE3.Rz(s[2])

class UnicycleAgent:
    """
    Combines Physics (roboticstoolbox) and Graphics (spatialgeometry).
    acts as the 'Digital Twin' of the robot.
    """
    def __init__(self, n, s, c) -> None:
        self.name = n
        # The kinematic model (handles the math of how wheels move the robot)
        self._robot = rtb.Unicycle()
        self._robot.init(np.array(s, dtype=float)) # Set initial state
        self._visual = make_agent(c)

    def register(self, e: swift.Swift) -> None:
        """Adds the robot visual to the Swift environment."""
        e.add(self._visual.body)
        self.update_visual()

    def state(self) -> np.ndarray:
        """Returns current [x, y, theta]."""
        return np.asarray(self._robot.q, dtype=float)

    def update_visual(self) -> None:
        """Syncs the visual block with the internal mathematical state."""
        self._visual.update(state_to_pose(self.state()))

    def apply_velocity_cmd(self, c) -> None:
        """
        Integrates the physics.
        Args: c = [v, omega] (Linear and Angular velocity)
        """
        self._robot.step(np.array(c, dtype=float)) # Physics step
        self.update_visual()                       # Graphics step

def add_wall(e, cx, cy, w, h) -> sg.Cuboid:
    """Helper function to add a static wall to the environment."""
    wall = sg.Cuboid(scale=[w, h, 0.5], color=COLOR_WALL)
    wall.T = sm.SE3(cx, cy, 0.25)
    e.add(wall)
    return wall


### --- SENSOR SIMULATION (RAYCASTING) ---


In [4]:
@dataclass
class SensorSim:
    """
    Simulates LIDAR/Ultrasonic sensors using Raycasting.
    Calculates the distance from the robot center to the nearest wall 
    along specific angles.
    """
    walls: List[sg.Cuboid]
    max_range: float = 5.0
    # Define sensor directions relative to robot heading (0 = Front)
    sensor_angles_rad: Dict[str, float] = field(default_factory=lambda: {
        'front': 0.0, 
        'front_left': np.pi/4,   # +45 degrees
        'front_right': -np.pi/4  # -45 degrees
    })
    
    def _ray_aabb_intersection(self, ro, rd, wall) -> float:
        """
        Calculates intersection between a Ray and an Axis-Aligned Bounding Box (Wall).
        standard algorithm in computer graphics for collision detection.
        
        Args:
            ro: Ray Origin [x, y]
            rd: Ray Direction vector [dx, dy]
            wall: The wall object to check against
        Returns:
            Distance to intersection, or max_range if no hit.
        """
        # Extract wall position (c) and size (s)
        c = wall.T[0:3, 3]
        s = wall.scale
        # Calculate Min (mb) and Max (Mb) corners of the wall
        mb, Mb = c - s / 2.0, c + s / 2.0
        
        t_min, t_max = 0.0, self.max_range
        
        # Check intersection for X and Y axes
        for i in range(2):
            if abs(rd[i]) < 1e-6: # Ray is parallel to the axis
                if ro[i] < mb[i] or ro[i] > Mb[i]: return self.max_range
            else:
                # Slab method for AABB intersection
                t1 = (mb[i] - ro[i]) / rd[i]
                t2 = (Mb[i] - ro[i]) / rd[i]
                t_n = min(t1, t2)
                t_f = max(t1, t2)
                t_min = max(t_min, t_n)
                t_max = min(t_max, t_f)
                
                if t_min > t_max or t_max < 0: return self.max_range
                
        return min(t_min, self.max_range) if t_min > 0 else self.max_range

    def get_readings(self, s: Sequence[float]) -> Dict[str, float]:
        """
        Main public method. Returns the distance readings for all sensors.
        Args: s = Robot state [x, y, theta]
        """
        x, y, theta = s
        ro = np.array([x, y]) # Ray Origin
        readings = {}
        
        for name, angle in self.sensor_angles_rad.items():
            # Global Angle of the sensor = Robot Heading + Sensor Mount Angle
            ga = theta + angle
            rd = np.array([math.cos(ga), math.sin(ga)]) # Ray Direction Unit Vector
            
            min_dist = self.max_range
            # Check against every wall and find the closest one
            for wall in self.walls: 
                dist = self._ray_aabb_intersection(ro, rd, wall)
                min_dist = min(min_dist, dist)
                
            readings[name] = min_dist
        return readings


### --- COSMETIC VISUALIZERS (SENSOR RAYS + LINE) ---


In [17]:
def make_ray_visual(c): 
    """Creates a thin yellow stick to represent a laser ray."""
    return sg.Cuboid(scale=[1.0, 0.02, 0.02], color=c)

def update_ray_visuals(rp, r, sa, vr):
    """Updates the length and rotation of the yellow sticks based on sensor data."""
    for name, ray_visual in vr.items():
        dist = r.get(name, 0.0)
        angle = sa.get(name, 0.0)
        ray_visual.scale[0] = dist # Stretch the stick to match distance
        # Complex transform: Robot Pose * Sensor Rotation * Shift stick center forward
        ray_visual.T = rp * sm.SE3.Rz(angle) * sm.SE3(dist / 2.0, 0, 0.05)

def add_path_visual(env: swift.Swift, waypoints: List[Tuple[float, float]]) -> None:
    """Draws grey static lines connecting the waypoints."""
    for i in range(len(waypoints) - 1):
        p1, p2 = waypoints[i], waypoints[i+1]
        dx, dy = p2[0] - p1[0], p2[1] - p1[1]
        length = np.hypot(dx, dy)
        angle = np.arctan2(dy, dx)
        cx, cy = (p1[0] + p2[0]) / 2.0, (p1[1] + p2[1]) / 2.0
        
        line = sg.Cuboid(scale=[length, 0.03, 0.01], color=COLOR_LINE)
        line.T = sm.SE3(cx, cy, 0.01) * sm.SE3.Rz(angle)
        env.add(line)


## --- MAIN SIMULATION LOOP ---


New version by Carlos, using setup environment containing a line in space and randomly placed "wall blocks", instead of walls forming a corridor.

In [6]:
# Simulation timing settings
dt = 0.05
simulation_time = 40.0
simulation_steps = int(simulation_time / dt)


### 1. ENTORNO EN EL PRIMER CUADRANTE + LÍNEA y = a x + b

In [7]:
# Solo trabajamos en coordenadas positivas
x_min, x_max = 0.0, 8.0
y_min, y_max = 0.0, 8.0

rng = np.random.default_rng()

# Elegimos a > 0 y b > 0 de forma que la línea quede dentro del área útil
# y tenga longitud suficiente en X
while True:
    a = float(rng.uniform(0.1, 1.0))   # pendiente positiva
    b = float(rng.uniform(0.5, 2.0))   # intercepto positivo (línea delante del robot)

    # x máximo para que y = a x + b no salga por arriba del área
    max_x_line = (y_max - b) / a
    max_x_line = min(x_max, max_x_line)

    # Queremos un segmento de línea con cierta longitud (> 2 m por ejemplo)
    if max_x_line > 2.0:
        break

# Construimos la trayectoria como una lista de waypoints sobre la línea
num_waypoints = 50
xs = np.linspace(0.0, max_x_line, num_waypoints)
WAYPOINTS = [(float(x), float(a * x + b)) for x in xs]

# El robot SIEMPRE empieza en el inicio de la línea (no aleatorio)
start_x, start_y = WAYPOINTS[0]

# Dirección de la línea (del primer al último waypoint)
line_vec = np.array(
    [WAYPOINTS[-1][0] - WAYPOINTS[0][0],
        WAYPOINTS[-1][1] - WAYPOINTS[0][1]],
    dtype=float,
)
line_vec /= np.linalg.norm(line_vec)

# Orientamos el robot mirando en la dirección de avance de la línea
start_theta = math.atan2(line_vec[1], line_vec[0])
start_state = [start_x, start_y, start_theta]

print(f"[ENV] Line: y = {a:.3f} x + {b:.3f}")
print(f"[ENV] Start state: x={start_x:.2f}, y={start_y:.2f}, theta={start_theta:.2f} rad")


[ENV] Line: y = 0.746 x + 1.313
[ENV] Start state: x=0.00, y=1.31, theta=0.64 rad


### 2. LANZAR SWIFT (SIN PAREDES DE ENTORNO)


In [25]:
env = swift.Swift()
env.launch(realtime=True, comms="rtc", browser="notebook")  # abre el navegador a mano si hace falta
# So as not to constantly have to maneuver camera, set it up above simulation area
env.set_camera_pose(position=[4.0, -2.0, 7.0], look_at=[4.0, 2.0, 3.0])

# time.sleep(1.0)


### 3. GENERAR OBSTÁCULOS SOLO DELANTE DEL ROBOT


In [26]:
walls: List[sg.Cuboid] = []

# Vector normal unitario a la línea (para desplazar obstáculos "cerca" de ella)
normal_vec = np.array([-line_vec[1], line_vec[0]])
max_line_distance = 0.8   # distancia máxima a la línea
num_obstacles = 6
obstacle_size_x = 0.5
obstacle_size_y = 0.5

for i in range(num_obstacles):
    # Elegimos un waypoint ADELANTE del origen de la línea (índice >= 1)
    idx = int(rng.integers(low=1, high=num_waypoints))
    bx, by = WAYPOINTS[idx]

    # La mitad de los obstáculos exactamente sobre la línea, la otra mitad cerca
    if i % 2 == 0:
        offset = 0.0
    else:
        offset = float(rng.uniform(-max_line_distance, max_line_distance))

    cx = bx + offset * normal_vec[0]
    cy = by + offset * normal_vec[1]

    # Aseguramos que todo está en el primer cuadrante y dentro del área útil
    if not (x_min <= cx <= x_max and y_min <= cy <= y_max):
        continue

    walls.append(add_wall(env, cx, cy, obstacle_size_x, obstacle_size_y))

print(f"[ENV] Generated {len(walls)} obstacles along/near the line.")


[ENV] Generated 6 obstacles along/near the line.


### 4. CONFIGURAR AGENTE, SENSORES Y CONTROLADOR HÍBRIDO


In [27]:
# Robot (modelo unicycle + visual)
robot_agent = UnicycleAgent("robot", start_state, COLOR_BLUE)

# Sensores de distancia (usan la lista 'walls' como obstáculos)
sensor_sim = SensorSim(walls=walls, max_range=3.0)

    # Controlador híbrido (planner + reactivo)
# Robot behavior mode:
#   "catcher"               -> chases the carrot
#   "perfect_line_follower" -> follows the line as perfectly as possible
mode = "perfect_line_follower"  # or "perfect_line_follower or catcher"

# Hybrid controller (planner + reactive layer)
controller = HybridNavigator(
    # Planner parameters
    waypoints=WAYPOINTS,
    reference_speed=0.8,
    linear_velocity_gain=2.0,
    angular_velocity_gain=4.0,

    # Reactive safeguard parameters
    reactive_angular_gain=2.5,
    safety_distance=1.0,
    wall_follow_distance=0.5,
    reactive_speed=0.4,

    # Physical limits
    max_linear_velocity=1.0,

    # NEW: behavior mode
    mode=mode,
)



[TrajectoryTracker] Path initialized.
  Total distance: 9.98m
  Total time (at v_ref=0.8): 12.47s


### 5. VISUALES (RAYOS + LÍNEA + MARCADOR DE REFERENCIA)


In [28]:
viz_rays = {name: make_ray_visual(COLOR_RAY) for name in sensor_sim.sensor_angles_rad}

ref_marker = sg.Sphere(radius=0.08, color=COLOR_GREEN)

robot_agent.register(env)
add_path_visual(env, WAYPOINTS)
env.add(ref_marker)
for ray_viz in viz_rays.values():
    env.add(ray_viz)

# Frame inicial
env.step(0.0)

# Logs
time_log, x_log, y_log = [], [], []
v_cmd_log, w_cmd_log = [], []
tracking_error_log = []


### 6. BUCLE PRINCIPAL


In [29]:
for step in range(simulation_steps):
    t = step * dt
    r_state = robot_agent.state()  # [x, y, theta]

    # A. SENSE: distancias a obstáculos
    readings = sensor_sim.get_readings(r_state)

    # B. PLAN & DECIDE: controlador híbrido
    v_cmd, w_cmd = controller.compute_control(
        r_state,
        t=t,
        d_front=readings["front"],
        d_front_left=readings["front_left"],
        d_front_right=readings["front_right"],
    )

    # C. ACT: aplicar velocidades al robot
    robot_agent.apply_velocity_cmd([v_cmd, w_cmd])

    # Actualizar rayos y marcador de referencia
    update_ray_visuals(
        state_to_pose(r_state),
        readings,
        sensor_sim.sensor_angles_rad,
        viz_rays,
    )

    ref_x, ref_y = controller._compute_reference_point(t)
    ref_marker.T = sm.SE3(ref_x, ref_y, 0.01)

    # D. LOG
    time_log.append(t)
    x_log.append(r_state[0])
    y_log.append(r_state[1])
    v_cmd_log.append(v_cmd)
    w_cmd_log.append(w_cmd)
    tracking_error_log.append(np.hypot(ref_x - r_state[0], ref_y - r_state[1]))

    # Cuando el planner termina la trayectoria, salimos
    if controller.is_done():
        print(f"Path completed at t={t:.2f}s.")
        break

    # Avanzar el reloj del simulador
    env.step(dt)

time.sleep(1.0)
env.close()

Path completed at t=12.50s.


## PLOT THE RESULTS


In [30]:

print("Simulation ended. Generating plots...")

time_log = np.array(time_log)
fig, axs = plt.subplots(3, 1, figsize=(10, 18))
fig.suptitle("Hybrid Navigation Analysis", fontsize=16)

# Plot 1: XY Trajectory
axs[0].plot(x_log, y_log, "b-", label="Actual Robot Path")
wp_x = [p[0] for p in WAYPOINTS]
wp_y = [p[1] for p in WAYPOINTS]
axs[0].plot(wp_x, wp_y, "g--", linewidth=2, label="Planner Trajectory (Desired)")
axs[0].plot(start_state[0], start_state[1], "go", markersize=10, label="Start")

# Dibujar obstáculos en el plano (ya no son "walls" de entorno)
for wall in walls:
    c = wall.T[0:3, 3]
    s = wall.scale
    xm, xM = c[0] - s[0] / 2, c[0] + s[0] / 2
    ym, yM = c[1] - s[1] / 2, c[1] + s[1] / 2
    axs[0].fill(
        [xm, xM, xM, xm],
        [ym, ym, yM, yM],
        color="r",
        alpha=0.7,
        label="Obstacles",
    )

# Legend sin duplicados
handles, labels = axs[0].get_legend_handles_labels()
by_label = dict(zip(labels, handles))
axs[0].legend(by_label.values(), by_label.keys(), loc="upper left")

axs[0].set_xlabel("X [m]")
axs[0].set_ylabel("Y [m]")
axs[0].set_title("Robot Path vs Plan")
axs[0].grid(True)
axs[0].axis("equal")
axs[0].set_xlim(x_min, x_max)
axs[0].set_ylim(y_min, y_max)

# Plot 2: Control Inputs (v, w) over time
axs[1].plot(time_log, v_cmd_log, "b", label="Lin. Velocity (v)")
axs[1].axhline(
    y=controller._v_ref,
    color="c",
    linestyle="--",
    label="Ref Speed",
)
axs[1].set_ylabel("v [m/s]")
axs[1].legend(loc="upper left")
axs[1].grid(True)

ax2 = axs[1].twinx()
ax2.plot(time_log, w_cmd_log, "m", label="Ang. Velocity (w)")
ax2.set_ylabel("w [rad/s]")
ax2.legend(loc="upper right")
axs[1].set_title("Control Inputs")

# Plot 3: Tracking Error
axs[2].plot(time_log, tracking_error_log, "r-", label="Tracking Error")
axs[2].set_xlabel("Time [s]")
axs[2].set_ylabel("Error [m]")
axs[2].grid(True)
axs[2].set_title("Tracking Error (Peaks indicate Obstacle Avoidance active)")
axs[2].legend()

plt.tight_layout(rect=[0, 0.03, 1, 0.95])
plt.savefig("hybrid_navigation_results.png", dpi=150)
print("Plots saved to hybrid_navigation_results_notebook.png")


Ignoring fixed x limits to fulfill fixed data aspect with adjustable data limits.
Ignoring fixed y limits to fulfill fixed data aspect with adjustable data limits.


Simulation ended. Generating plots...
Plots saved to hybrid_navigation_results_notebook.png
